# Ön çalışmayı (pilot) Colab'da yeniden çalıştırma

Bu defter `pilot` klasöründeki deneyi Google Colab'da baştan sona yeniden çalıştırır:

1. modelleri indirir,
2. modellere İngilizce ve Türkçe istemle kod ve test yazdırır, iki azaltım stratejisini (ara çeviri, iki dilli istem) dener,
3. sonuçları değerlendirip `results/OZET.md` özetini üretir,
4. senin sonuçlarını bulut oturumunda üretilen sonuçlarla karşılaştırır.

**Başlamadan önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**. GPU olmadan da çalışır ama saatler sürer.

**Güvenlik:** Modelin yazdığı kod bu oturumda, izole ve zaman sınırlı bir süreçte çalıştırılır. Colab geçici bir sanal makine olduğu için bu iş kişisel bilgisayardan daha güvenlidir.

**Not:** Bu defter bulut oturumunda hazırlandı ve Colab'da henüz denenmedi. Bir hücre hata verirse hata mesajını not al. Sorunun çoğu kurulum adımından çıkar.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU yok: CPU ile devam edilecek (yavaş)"

## 1. Depoyu indir

Depo GitHub'dan indirilir. İnternet erişimi yoksa aşağıdaki hücrede `KLON = False` yapıp bilgisayarındaki `pilot.zip` dosyasını yükleyebilirsin (zip, `pilot/` klasörünü içermeli).

In [ ]:
KLON = True
if KLON:
    !git clone -q https://github.com/EmiRZx/turkce-istem-kod-pilot /content/pilot
else:
    from google.colab import files
    yuklenen = files.upload()  # pilot.zip
    !unzip -q -o pilot.zip -d /content
!ls /content/pilot
%cd /content/pilot

## 2. Kurulum

`llama-cpp-python` GPU desteğiyle kaynaktan derlenir. Bu adım 10–20 dakika sürebilir; çıktı vermeden beklemesi normaldir. Sürüm, bulut oturumunda kullanılanla aynıdır.

In [ ]:
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
env = dict(os.environ)
if gpu:
    env["CMAKE_ARGS"] = "-DGGML_CUDA=on"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python==0.3.36", "scipy", "pytest"],
               env=env, check=True)
os.environ["GPU_LAYERS"] = "-1" if gpu else "0"   # -1: tum katmanlar GPU'da
print("GPU:", gpu, "| GPU_LAYERS =", os.environ["GPU_LAYERS"])

## 3. Modelleri indir (yaklaşık 1,1 GB + 4,7 GB)

In [ ]:
!mkdir -p models
!cd models && curl -sSL -O https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf
!cd models && curl -sSL -O https://huggingface.co/Qwen/Qwen2.5-Coder-7B-Instruct-GGUF/resolve/main/qwen2.5-coder-7b-instruct-q4_k_m.gguf
!ls -lh models

## 4. Hızlı deneme (3 problem)

Kurulumun çalıştığını görmek için önce üç problemle küçük bir deneme yapılır. Sonuç ayrı bir klasöre yazılır.

In [ ]:
!python3 -I src/run_codegen.py --model models/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf --name deneme --lang tr \
    --data data/humaneval_en_tr.jsonl --out deneme/codegen_deneme_tr.jsonl --limit 3
!python3 -I src/evaluate_codegen.py data/humaneval_en_tr.jsonl "deneme/codegen_*.jsonl" --out deneme/eval.jsonl
!cat deneme/eval.jsonl

## 5. Pilotu baştan çalıştır

Zip'teki `results/` klasörü bulut oturumunun sonuçlarını içerir. Betikler kaldığı yerden devam ettiği için, sıfırdan üretmek üzere önce bu klasör `results_bulut/` adıyla kenara alınır. Sonra karşılaştırmada kullanılacak.

**Süre:** 4 çekirdekli CPU'da toplam yaklaşık 12 saat sürer (bulut oturumunda parça parça çalıştırıldı). T4 GPU'da çok daha kısa sürmesi beklenir ama denenmedi. Hücre yarıda kalırsa (bağlantı koparsa) aynı hücreyi yeniden çalıştır: betikler bitmiş problemleri atlar. Colab çalışma zamanı tamamen sıfırlanırsa dosyalar silinir; uzun çalıştırmalarda `results/` klasörünü ara ara indir ya da Google Drive'a kopyala.

In [ ]:
!test -d results_bulut || mv results results_bulut
!PY=python3 MODELS=models bash run_pilot.sh 2>&1 | tail -n 5

## 6. Değerlendirme ve özet

In [ ]:
!python3 -I src/evaluate_codegen.py data/humaneval_en_tr.jsonl "results/raw/codegen_*.jsonl" --out results/codegen_eval.jsonl
!python3 -I src/evaluate_testgen.py data/humaneval_en_tr.jsonl "results/raw/testgen_*.jsonl" --out results/testgen_eval.jsonl
!python3 -I src/analyze.py --codegen results/codegen_eval.jsonl --testgen results/testgen_eval.jsonl \
    --data data/humaneval_en_tr.jsonl --subset data/subset_60.txt --out-md results/OZET.md --out-json results/summary.json > /dev/null
!python3 -I src/evaluate_codegen.py data/humaneval_en_tr.jsonl "results/raw_azaltim/codegen_*.jsonl" --out results/azaltim_eval.jsonl
!python3 -I src/analyze_mitigation.py --codegen results/codegen_eval.jsonl --mitigation results/azaltim_eval.jsonl \
    --translations "results/raw_azaltim/ceviri_*.jsonl" --out-md results/AZALTIM.md --out-json results/azaltim_summary.json > /dev/null
from IPython.display import Markdown, display
display(Markdown(open("results/OZET.md", encoding="utf-8").read()))
display(Markdown(open("results/AZALTIM.md", encoding="utf-8").read()))

## 7. Bulut sonuçlarıyla karşılaştırma

Açgözlü çözümleme bile tam olarak tekrarlanabilir değil: bulut oturumunda 1,5B modelin kod üretimi aynı makinede ikinci kez çalıştırıldığında geçme/kalma sonucu problemlerin yaklaşık %96'sında aynı kaldı, pass@1 1 puan kadar oynadı ve EN–TR farkı 19,5 yerine 18,9 puan çıktı (`results_bulut/TEKRAR.md`). GPU ile CPU arasındaki sayısal farklar bunu artırabilir. Bu yüzden birkaç problemde farklı sonuç çıkması normaldir. Önemli olan, EN–TR farkının yönünün ve büyüklüğünün korunmasıdır.

In [ ]:
import json
def oku(yol):
    return {(r["model"], r["lang"], r["task_id"]): r["passed"]
            for r in map(json.loads, open(yol, encoding="utf-8"))}
yeni, eski = oku("results/codegen_eval.jsonl"), oku("results_bulut/codegen_eval.jsonl")
yeni.update(oku("results/azaltim_eval.jsonl"))
eski.update(oku("results_bulut/azaltim_eval.jsonl"))
ortak = sorted(set(yeni) & set(eski))
for model in sorted({k[0] for k in ortak}):
    for dil in ("en", "tr", "tr_araceviri", "tr_ikidilli"):
        ks = [k for k in ortak if k[0] == model and k[1] == dil]
        if not ks:
            continue
        ayni = sum(yeni[k] == eski[k] for k in ks)
        yuzde = lambda d: f"{100 * sum(d[k] for k in ks) / len(ks):.1f}".replace(".", ",")
        print(f"{model} {dil.upper()}: {len(ks)} problemden {ayni} tanesinde sonuç aynı | "
              f"pass@1 bulut %{yuzde(eski)}, Colab %{yuzde(yeni)}")

## 8. Sonuçları indir

In [ ]:
from google.colab import files
!zip -q -r pilot_sonuclar_colab.zip results deneme
files.download("pilot_sonuclar_colab.zip")